# Superstore Sales Analysis — Executive Revenue Dashboard
**End-to-end analytics project | Python · PySpark · Databricks**

---

## Business context

This notebook analyses three years of retail sales data (2021–2023) across four regions, three product categories, and three customer segments. The objective is to answer one question:

> **Where is this business leaking profit, and what should leadership prioritise?**

The analysis follows a CFO-ready narrative structure:
1. Overall financial performance
2. What is driving (and destroying) profit margins
3. Where discounting is being misused
4. Which segments and regions deserve more investment
5. Executive recommendations


## 1. Environment setup

Install and import all required libraries. This notebook runs on **Databricks** (PySpark available natively) and locally via standard Python.


In [ ]:
# Standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# PySpark (available natively on Databricks)
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Initialise Spark session (auto-created on Databricks; explicit here for local runs)
spark = SparkSession.builder.appName('SuperstoreSalesAnalysis').getOrCreate()

print('Spark version:', spark.version)
print('Setup complete.')

## 2. Data loading and schema validation

Load the dataset into a Spark DataFrame for scalable processing, then convert to pandas for visualisation.


In [ ]:
# Load CSV into Spark DataFrame
sdf = spark.read.csv('superstore_sales.csv', header=True, inferSchema=True)

# Inspect schema
sdf.printSchema()
print(f'Total records: {sdf.count():,}')

In [ ]:
# Convert to pandas for analysis and visualisation
df = sdf.toPandas()
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['Year'] = df['Order_Date'].dt.year
df['Month'] = df['Order_Date'].dt.month
df['Profit_Margin_Pct'] = (df['Profit'] / df['Sales'] * 100).round(2)

print('Shape:', df.shape)
df.head()

## 3. Exploratory data analysis

Quick health check before diving into business questions.


In [ ]:
# Missing values
print('Missing values per column:')
print(df.isnull().sum())

# Summary stats
print('\nKey financial metrics:')
print(f"  Total Revenue : €{df['Sales'].sum():>12,.0f}")
print(f"  Total Profit  : €{df['Profit'].sum():>12,.0f}")
print(f"  Overall Margin: {df['Profit'].sum()/df['Sales'].sum()*100:>11.1f}%")
print(f"  Total Orders  : {df.shape[0]:>12,}")
print(f"  Date Range    :  {df['Order_Date'].min().date()} → {df['Order_Date'].max().date()}")

## 4. Revenue and profit performance

### 4.1 Year-on-year financial overview


In [ ]:
yearly = df.groupby('Year').agg(
    Revenue=('Sales', 'sum'),
    Profit=('Profit', 'sum')
).reset_index()
yearly['Margin'] = (yearly['Profit'] / yearly['Revenue'] * 100).round(1)
yearly['Revenue_M'] = (yearly['Revenue'] / 1e6).round(2)
yearly['Profit_M'] = (yearly['Profit'] / 1e6).round(2)

print(yearly[['Year','Revenue_M','Profit_M','Margin']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(yearly)); w = 0.35
b1 = ax.bar(x - w/2, yearly['Revenue']/1e6, w, label='Revenue', color='#1B5E20', alpha=0.85)
b2 = ax.bar(x + w/2, yearly['Profit']/1e6,  w, label='Profit',  color='#F9A825', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(yearly['Year'])
ax.set_ylabel('€ Millions')
ax.set_title('Revenue and profit by year', fontweight='bold')
ax.legend(); ax.spines[['top','right']].set_visible(False)
for b in b1: ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f'€{b.get_height():.2f}M', ha='center', fontsize=9)
for b in b2: ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f'€{b.get_height():.2f}M', ha='center', fontsize=9)
plt.tight_layout(); plt.savefig('chart1_revenue_profit_year.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.2 Monthly revenue trend (seasonality)


In [ ]:
monthly = df.groupby(['Year','Month'])['Sales'].sum().reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
for yr, grp in monthly.groupby('Year'):
    ax.plot(grp['Month'], grp['Sales']/1e3, marker='o', markersize=4, label=str(yr), linewidth=2)
ax.set_xticks(range(1,13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_ylabel('Revenue (€000s)')
ax.set_title('Monthly revenue trend — consistent Q4 spike across all years', fontweight='bold')
ax.legend(title='Year'); ax.spines[['top','right']].set_visible(False)
ax.grid(alpha=0.3, linestyle='--')
plt.tight_layout(); plt.savefig('chart5_monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

# CFO insight
print('► Q4 (Oct–Dec) consistently accounts for the highest revenue across all three years.')
print('► Inventory and staffing planning should be calibrated to this pattern.')

## 5. Profitability deep-dive

### 5.1 Profit margin by sub-category

Not all products contribute equally. This is where the real story is.


In [ ]:
sub_perf = df.groupby('Sub_Category').agg(
    Sales=('Sales','sum'), Profit=('Profit','sum')
).reset_index()
sub_perf['Margin'] = (sub_perf['Profit'] / sub_perf['Sales'] * 100).round(1)
sub_perf = sub_perf.sort_values('Margin')

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#C62828' if m < 0 else '#1B5E20' if m > 15 else '#455A64' for m in sub_perf['Margin']]
bars = ax.barh(sub_perf['Sub_Category'], sub_perf['Margin'], color=colors, alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Profit margin (%)')
ax.set_title('Profit margin by sub-category — tables are destroying value', fontweight='bold')
ax.spines[['top','right']].set_visible(False)
for bar, val in zip(bars, sub_perf['Margin']):
    ax.text(val + (0.3 if val >= 0 else -0.3), bar.get_y()+bar.get_height()/2,
            f'{val}%', va='center', ha='left' if val >= 0 else 'right', fontsize=9)
plt.tight_layout(); plt.savefig('chart2_margin_subcategory.png', dpi=150, bbox_inches='tight')
plt.show()

worst = sub_perf.iloc[0]
best  = sub_perf.iloc[-1]
print(f'► Worst margin: {worst.Sub_Category} at {worst.Margin}% — actively destroying profit')
print(f'► Best margin:  {best.Sub_Category} at {best.Margin}%')

### 5.2 Profit heatmap — region × category


In [ ]:
pivot = df.pivot_table(values='Profit', index='Region', columns='Category', aggfunc='sum')
pivot_k = pivot.div(1000).round(1)

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(pivot_k, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Profit (€000s)'})
ax.set_title('Profit by region and category (€000s)\nFurniture underperforms in every region', fontweight='bold')
ax.set_xlabel(''); ax.set_ylabel('')
plt.tight_layout(); plt.savefig('chart4_region_category_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('► Furniture generates negative or near-zero profit in Central and South regions.')
print('► Technology is the most profitable category in every region.')

## 6. Discount analysis — the profit leak

Discounts are the single most controllable lever for improving margins.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
cat_colors = {'Technology': '#1B5E20', 'Furniture': '#F9A825', 'Office Supplies': '#1565C0'}
sample = df.sample(800, random_state=42)
for cat, grp in sample.groupby('Category'):
    ax.scatter(grp['Discount'], grp['Profit'], alpha=0.4, s=18, color=cat_colors[cat], label=cat)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Discount applied')
ax.set_ylabel('Profit (€)')
ax.set_title('Higher discounts consistently destroy profit', fontweight='bold')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend(title='Category'); ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.savefig('chart3_discount_profit.png', dpi=150, bbox_inches='tight')
plt.show()

# Quantify the damage
disc_impact = df[df['Discount'] > 0]['Profit'].sum() - df[df['Discount'] == 0]['Profit'].sum()
high_disc = df[df['Discount'] >= 0.3]
print(f'► Orders with discount ≥ 30%: {len(high_disc):,} orders')
print(f'► Total profit on those orders: €{high_disc["Profit"].sum():,.0f}')
print(f'► Average margin on discounted orders: {high_disc["Profit"].sum()/high_disc["Sales"].sum()*100:.1f}%')

## 7. Customer segment analysis


In [ ]:
seg = df.groupby('Segment').agg(Sales=('Sales','sum'), Profit=('Profit','sum')).reset_index()
seg['Margin'] = (seg['Profit']/seg['Sales']*100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
colors = ['#1B5E20','#388E3C','#81C784']
axes[0].bar(seg['Segment'], seg['Sales']/1e6, color=colors, alpha=0.85)
axes[0].set_ylabel('€ Millions'); axes[0].set_title('Revenue by segment')
axes[0].spines[['top','right']].set_visible(False)
for i,(_, row) in enumerate(seg.iterrows()):
    axes[0].text(i, row['Sales']/1e6+0.02, f'€{row["Sales"]/1e6:.2f}M', ha='center', fontsize=9)
axes[1].bar(seg['Segment'], seg['Margin'], color=colors, alpha=0.85)
axes[1].set_ylabel('Profit margin (%)'); axes[1].set_title('Profit margin by segment')
axes[1].spines[['top','right']].set_visible(False)
for i,(_, row) in enumerate(seg.iterrows()):
    axes[1].text(i, row['Margin']+0.2, f'{row["Margin"]}%', ha='center', fontsize=9)
plt.suptitle('Corporate: highest revenue and strongest margins', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.savefig('chart6_segment_profitability.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Executive recommendations

Based on the analysis, four actions are recommended:

### 1. Fix the Tables problem
Tables are the only sub-category with a **negative profit margin**. Every table sold loses money. Options: reprice, eliminate low-margin SKUs, or exit the segment.

### 2. Introduce a discount governance policy
Orders with discounts ≥ 30% are consistently loss-making. A hard cap at 20% with manager approval above 15% would materially improve overall margin without impacting volume.

### 3. Double down on Corporate segment
Corporate accounts are the highest-revenue and highest-margin segment. Targeted account management and volume incentives (not discounts) should be prioritised here.

### 4. Investigate Furniture in Central and South regions
Furniture is loss-making in these two regions specifically. This warrants a logistics/pricing audit — the issue may be shipping costs or localised discounting behaviour.

---

| Priority | Action | Expected impact |
|---|---|---|
| High | Cap discounts at 20% | +2–3% overall margin |
| High | Review Tables pricing/SKU mix | Eliminate negative-margin sales |
| Medium | Grow Corporate accounts | Highest ROI customer segment |
| Medium | Regional audit: Furniture (Central, South) | Recover €X0K profit leakage |


In [ ]:
# Final summary using Spark SQL — demonstrates Databricks capability
sdf.createOrReplaceTempView('sales')

summary = spark.sql("""
    SELECT
        Category,
        ROUND(SUM(Sales), 0)               AS total_revenue,
        ROUND(SUM(Profit), 0)              AS total_profit,
        ROUND(SUM(Profit)/SUM(Sales)*100, 1) AS margin_pct,
        COUNT(*)                           AS order_count
    FROM sales
    GROUP BY Category
    ORDER BY total_revenue DESC
""")
summary.show()
print('Analysis complete.')